# Object Schema Patch v0.2.1

**This is a bounded v0.2.1 schema patch, not a complete Data Audit.**

Không scan ID set, không decode video, không đọc image pixels, không chạy model/GPU.

In [ ]:
from pathlib import Path
import os, subprocess, sys

REPO_URL = os.environ.get('AIC_REPO_URL', 'https://github.com/Irthn1311/AIC2026_TeamPTK_SGU.git')
REPO_REF = os.environ.get('AIC_REPO_REF', 'TRIAGEEG')
REPO_DIR = Path('/kaggle/working/AIC2026_TeamPTK_SGU')
DATASET_ROOT = Path(os.environ.get('AIC_DATA_ROOT', '/kaggle/input/datasets/nadkli/dataset-aic'))
OUTPUT_ROOT = Path('/kaggle/working/cross_asset_survey_v021')
V02_SUMMARY = Path('/kaggle/working/cross_asset_survey_v02/cross_asset_survey_v02.json')
REFRESH_REPO = os.environ.get('AIC_REFRESH_REPO') == '1'
print('This is a bounded v0.2.1 schema patch, not a complete Data Audit.')

In [ ]:
def run_git(*args):
    completed = subprocess.run(['git', *args], cwd=REPO_DIR if REPO_DIR.exists() else None, text=True, capture_output=True)
    if completed.returncode:
        raise RuntimeError(completed.stderr.strip() or completed.stdout.strip())
    return completed.stdout.strip()

if (REPO_DIR / '.git').is_dir():
    if REFRESH_REPO:
        run_git('fetch', '--depth', '1', 'origin', REPO_REF)
        run_git('checkout', '--detach', 'FETCH_HEAD')
elif REPO_DIR.exists() and any(REPO_DIR.iterdir()):
    raise RuntimeError(f'{REPO_DIR} exists but is not a Git checkout')
else:
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(['git', 'clone', '--filter=blob:none', '--no-checkout', REPO_URL, str(REPO_DIR)], check=True)
    run_git('fetch', '--depth', '1', 'origin', REPO_REF)
    run_git('checkout', '--detach', 'FETCH_HEAD')
module_file = REPO_DIR / 'src/triage_eg/data/cross_asset_patch_v021.py'
if not module_file.is_file():
    raise RuntimeError(f'Missing {module_file}. Commit/push the v0.2.1 patch to {REPO_REF}, then set AIC_REFRESH_REPO=1 and rerun.')
sys.path.insert(0, str(REPO_DIR / 'src'))
print('git commit:', run_git('rev-parse', 'HEAD'))
print('python:', sys.version)

In [ ]:
from triage_eg.data.cross_asset_patch_v021 import PatchLimits, run_patch, write_outputs
limits = PatchLimits(max_object_json_total=15, max_object_json_bytes=1048576, max_boxes_per_file=20, max_boxes_total=100)
result = run_patch(DATASET_ROOT, limits=limits, v02_summary=V02_SUMMARY if V02_SUMMARY.is_file() else None, strict_root=True)
artifact_paths = write_outputs(result, OUTPUT_ROOT)
print(result.summary['disclaimer'])

In [ ]:
for sample in result.object_samples:
    lengths = {k: v['length'] for k, v in sample['field_observations'].items()}
    print(sample['video_id'], f"{sample['ordinal_n']:03d}", lengths, 'valid=', sample['parallel_arrays_valid'])

In [ ]:
schema = result.summary['object_schema_summary']
print({k: schema[k] for k in ('files_inspected','detection_count_min','detection_count_max','detection_count_total_in_sample')})

In [ ]:
print({k: schema[k] for k in ('bbox_item_length_distribution','sampled_coordinate_min','sampled_coordinate_max','coordinate_scale_hypothesis','bbox_order')})

In [ ]:
print('entities', schema['class_entity_item_types'])
print('labels', schema['class_label_item_types'])
print('names', schema['class_name_item_types'])
print('scores', schema['score_item_types'])

In [ ]:
for case in result.duplicate_cases:
    print(case['video_id'], case['frame_idx'], case['n_values'], case['all_expected_files_exist'])
    for asset in case['related_assets']: print(' ', asset)

In [ ]:
for label in ('verified_contracts','inferred_contracts','unknown_contracts'):
    print(label.upper())
    print(*result.summary[label], sep='\n')

In [ ]:
print(result.summary['readiness'])

In [ ]:
from zipfile import ZipFile
zip_path = artifact_paths['zip']
with ZipFile(zip_path) as archive:
    members = archive.namelist()
assert len(members) == 5 and 'cross_asset_survey_v021.zip' not in members
print('DOWNLOAD ZIP:', zip_path)
print('size_bytes:', zip_path.stat().st_size)
print('members:', members)